In [59]:
import pandas as pd
import numpy as np

In [60]:
df = pd.read_csv('Data/b2busers.csv')
df = df.drop(columns=['Unnamed: 0'])
df

,customerID,month,logins,churned
0,191126,-12,130,True
1,191126,-11,119,True
2,191126,-10,123,True
3,191126,-9,133,True
4,191126,-8,111,True
...,...,...,...,...
2395,909501,-5,138,False
2396,909501,-4,155,False
2397,909501,-3,127,False
2398,909501,-2,133,False


## Calcolo del segnale 
Da numerose osservazioni per cliente vogliamo avere un solo numero per cliente, il suo segnale. Che risponde alla domanda “ sta usando il prodotto meno di prima?” Credo valga anche Come sta usando il prodotto.

In [61]:
#filtro i mesi -12to-7,raggruppo per customer, media degli accessi
df_riferimento = df[df["month"].between(-12, -7)].groupby(['customerID', 'churned'], as_index=False)['logins'].mean()
df_riferimento = df_riferimento.rename(columns={"logins": "riferimento_mean"})

#filtro i mesi -6to-3,raggruppo per customer, media degli accessi
df_osservazione = df[df["month"].between(-6, -3)].groupby(['customerID', 'churned'], as_index=False)['logins'].mean()
df_osservazione = df_osservazione.rename(columns={"logins": "osservazione_mean"})

#unisco in un df
df_unito = df_riferimento.merge(df_osservazione, on="customerID")
df_unito = df_unito.rename(columns={"churned_y": "churned"})

#calcolo variazione di accessi per customer: (obs-reference)/reference * 100
df_unito['variazione'] = round((df_unito['osservazione_mean'] - df_unito['riferimento_mean'])/df_unito['riferimento_mean'] * 100,2)

df_unito = df_unito[['customerID', 'riferimento_mean','osservazione_mean', 'variazione', 'churned']]

df_unito

,customerID,riferimento_mean,osservazione_mean,variazione,churned
0,105407,188.166667,146.50,-22.14,False
1,106086,98.166667,101.00,2.89,False
2,110351,39.833333,40.50,1.67,False
3,116534,171.666667,182.50,6.31,False
4,121725,81.833333,88.25,7.84,False
...,...,...,...,...,...
195,986892,193.833333,190.25,-1.85,False
196,988571,44.000000,30.75,-30.11,False
197,989477,173.666667,179.25,3.21,False
198,998395,169.666667,166.75,-1.72,False


## Assegno score 
converto il segnale da valore continuo a uno score. Decido tramite deviazione standard la soglia che divide gli score

In [62]:
df_unito["risk"] = np.where(df_unito["variazione"] <= -20, "high", "low")

In [69]:
df_unito[df_unito['risk'] == 'high']['churned'].value_counts()
df_unito[df_unito['risk'] == 'low']['churned'].value_counts()

churned
False    161
True      24
Name: count, dtype: int64